In [ ]:
import os
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from copy        import deepcopy
from scipy       import stats
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn            as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

In [ ]:
INPUT_FILE = "../../data/processed/merged_features_weekly.csv"

TARGET   = "target_price"
LOOKBACK = 52
HORIZON  = 1

FEATURE_COLS = [
    "api_energy_and_lubricants",
    "api_fertilisers_and_soil_improvers",
    "api_plant_protection_products",
    "api_fresh_fruit",
    "api_fresh_vegetables",
    "fuel_petrol_price",
    "fuel_diesel_price",
    "sppi_road_freight",
    "shock_2021q4_2023q1",
    "post_shock",
    "week_sin",
    "week_cos",
]

D_MODEL  = 64
D_STATE  = 16
D_CONV   = 4
EXPAND   = 2
N_LAYERS = 3
DROPOUT  = 0.1

BATCH_SIZE = 32
EPOCHS     = 150
LR         = 1e-3
PATIENCE   = 20
SEED       = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
try:
    from mamba_ssm import Mamba as _OfficialMamba
    _HAVE_MAMBA_SSM = True
    print("mamba-ssm found — using optimised CUDA kernel")
except ImportError:
    _HAVE_MAMBA_SSM = False
    print("mamba-ssm not installed — using pure-PyTorch selective SSM")


class SelectiveSSM(nn.Module):
    def __init__(self, d_inner: int, d_state: int):
        super().__init__()
        self.d_inner = d_inner
        self.d_state = d_state
        self.x_proj  = nn.Linear(d_inner, d_state * 2 + d_inner, bias=False)
        self.dt_proj = nn.Linear(d_inner, d_inner, bias=True)
        A_log        = torch.arange(1, d_state + 1, dtype=torch.float32).log()
        self.A_log   = nn.Parameter(A_log.repeat(d_inner, 1))
        self.D       = nn.Parameter(torch.ones(d_inner))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, d = x.shape
        N = self.d_state
        xz    = self.x_proj(x)
        B_ssm = xz[..., :N]
        C_ssm = xz[..., N : 2 * N]
        dt    = F.softplus(self.dt_proj(xz[..., 2 * N :]))
        A     = -self.A_log.exp()
        A_bar = torch.exp(dt.unsqueeze(-1) * A)
        B_bar = dt.unsqueeze(-1) * B_ssm.unsqueeze(2)
        h  = torch.zeros(B, d, N, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            h  = A_bar[:, t] * h + B_bar[:, t] * x[:, t].unsqueeze(-1)
            yt = (h * C_ssm[:, t].unsqueeze(1)).sum(-1)
            ys.append(yt)
        return torch.stack(ys, dim=1) + self.D * x


class MambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        d_inner       = int(d_model * expand)
        self.norm     = nn.LayerNorm(d_model)
        self.in_proj  = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d   = nn.Conv1d(
            d_inner, d_inner, d_conv,
            padding=d_conv - 1, groups=d_inner, bias=True
        )
        self.ssm      = SelectiveSSM(d_inner, d_state)
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self._use_official = False
        if _HAVE_MAMBA_SSM:
            try:
                self._official     = _OfficialMamba(d_model=d_model, d_state=d_state,
                                                    d_conv=d_conv,   expand=expand)
                self._use_official = True
            except Exception:
                pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.norm(x)
        if self._use_official:
            return self._official(x) + residual
        xz      = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)
        x_c = self.conv1d(x_in.transpose(1, 2))
        x_c = x_c[..., : x_in.shape[1]].transpose(1, 2)
        x_c = F.silu(x_c)
        y   = self.ssm(x_c)
        y   = y * F.silu(z)
        return self.out_proj(y) + residual


class MambaForecaster(nn.Module):
    def __init__(self, n_vars, seq_len, horizon,
                 d_model=64, d_state=16, d_conv=4,
                 expand=2, n_layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_vars, d_model)
        self.blocks     = nn.ModuleList([
            MambaBlock(d_model, d_state, d_conv, expand)
            for _ in range(n_layers)
        ])
        self.dropout  = nn.Dropout(dropout)
        self.norm_out = nn.LayerNorm(d_model)
        self.head     = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, horizon),
        )

    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm_out(x)
        x = self.dropout(x)
        return self.head(x[:, -1, :])

In [ ]:
class TSDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, lookback: int, horizon: int):
        self.X, self.y   = X, y
        self.lookback    = lookback
        self.horizon     = horizon

    def __len__(self):
        return len(self.X) - self.lookback - self.horizon + 1

    def __getitem__(self, idx):
        x = self.X[idx : idx + self.lookback]
        y = self.y[idx + self.lookback : idx + self.lookback + self.horizon]
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
        )

In [ ]:
def train_epoch(model, loader, opt, criterion, device):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item() * len(xb)
    return total / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total, preds, actuals = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out    = model(xb)
        total += criterion(out, yb).item() * len(xb)
        preds.append(out.cpu().numpy())
        actuals.append(yb.cpu().numpy())
    return (
        total / len(loader.dataset),
        np.concatenate(preds).flatten(),
        np.concatenate(actuals).flatten(),
    )


@torch.no_grad()
def walk_forward(model, X_ctx, n_test, lookback, device):
    model.eval()
    preds = []
    for i in range(n_test):
        window = X_ctx[i : i + lookback]
        xb     = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)
        preds.append(model(xb).cpu().numpy().flatten()[0])
    return np.array(preds)

In [ ]:
def compute_metrics(actual, pred, train, label):
    a, p      = actual.flatten(), pred.flatten()
    naive_mae = np.mean(np.abs(np.diff(train)))
    return {
        "model": label,
        "RMSE" : round(float(np.sqrt(np.mean((a - p) ** 2))),                         4),
        "MAE"  : round(float(np.mean(np.abs(a - p))),                                 4),
        "MASE" : round(float(np.mean(np.abs(a - p)) / naive_mae),                     4),
        "sMAPE": round(float(np.mean(200 * np.abs(a - p) / (np.abs(a) + np.abs(p)))), 4),
        "MAPE" : round(float(np.mean(np.abs((a - p) / a)) * 100),                     4),
    }

In [ ]:
def show_training_curve(tr_losses, vl_losses, best_ep, label):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(tr_losses, color="#2c3e50", linewidth=1.5, label="Train")
    ax.plot(vl_losses, color="#e74c3c", linewidth=1.5, label="Validation")
    ax.axvline(best_ep, color="#27ae60", linestyle="--",
               label=f"Best epoch ({best_ep})")
    ax.set_title(f"Training Curve — {label}", fontweight="bold", fontsize=13)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
    plt.tight_layout()
    plt.show()


def show_forecast_plot(dates_train, train_vals,
                       dates_test,  test_vals, pred_vals, label):
    cutoff = dates_train[-1] - pd.Timedelta(weeks=104)
    mask   = dates_train >= cutoff
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(dates_train[mask], train_vals[mask],
            color="#bdc3c7", linewidth=0.9, label="Train")
    ax.plot(dates_test, test_vals,
            color="#2c3e50", linewidth=1.2, label="Actual")
    ax.plot(dates_test, pred_vals,
            color="#e74c3c", linewidth=1.2, linestyle="--", label="Mamba forecast")
    ax.set_title(f"Mamba Forecast — {label}", fontweight="bold", fontsize=13)
    ax.set_ylabel("Price")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()


def show_residual_diagnostics(residuals, fitted_vals, label):
    resid = residuals
    ci    = 1.96 / np.sqrt(len(resid))

    fig = plt.figure(figsize=(14, 12))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(resid, color="#2c3e50", linewidth=0.6)
    ax1.axhline(0, color="red", linestyle="--")
    ax1.set_title("Residuals over time")
    ax1.set_xlabel("Index"); ax1.set_ylabel("Residual")

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(resid, bins=40, density=True, color="#3498db", alpha=0.7, edgecolor="white")
    xr  = np.linspace(resid.min(), resid.max(), 200)
    ax2.plot(xr, stats.norm.pdf(xr, resid.mean(), resid.std()),
             color="red", linewidth=1.5)
    ax2.set_title("Residual distribution")
    ax2.set_xlabel("Residual"); ax2.set_ylabel("Density")

    ax3  = fig.add_subplot(gs[1, 0])
    lags = min(52, len(resid) // 2 - 1)
    acf  = [np.corrcoef(resid[:-k], resid[k:])[0, 1] for k in range(1, lags + 1)]
    ax3.stem(range(1, lags + 1), acf, basefmt=" ", linefmt="#2980b9", markerfmt="o")
    ax3.axhline( ci, color="red", linestyle="--")
    ax3.axhline(-ci, color="red", linestyle="--")
    ax3.axhline(0, color="black", linewidth=0.6)
    ax3.set_title("ACF of residuals")
    ax3.set_xlabel("Lag"); ax3.set_ylabel("ACF")

    ax4 = fig.add_subplot(gs[1, 1])
    try:
        from statsmodels.tsa.stattools import pacf as sm_pacf
        pv = sm_pacf(resid, nlags=lags, method="ywmle")
        ax4.stem(range(1, len(pv)), pv[1:], basefmt=" ",
                 linefmt="#8e44ad", markerfmt="o")
        ax4.axhline( ci, color="red", linestyle="--")
        ax4.axhline(-ci, color="red", linestyle="--")
        ax4.axhline(0, color="black", linewidth=0.6)
    except Exception:
        ax4.text(0.5, 0.5, "PACF unavailable", ha="center", va="center",
                 transform=ax4.transAxes)
    ax4.set_title("PACF of residuals")
    ax4.set_xlabel("Lag"); ax4.set_ylabel("PACF")

    ax5 = fig.add_subplot(gs[2, 0])
    (osm, osr), (slope, intercept, _) = stats.probplot(resid, dist="norm")
    ax5.scatter(osm, osr, alpha=0.4, s=8, color="#2c3e50")
    ax5.plot(osm, slope * np.array(osm) + intercept, color="red", linewidth=1.5)
    ax5.set_title("Q-Q plot")
    ax5.set_xlabel("Theoretical quantiles"); ax5.set_ylabel("Sample quantiles")

    ax6 = fig.add_subplot(gs[2, 1])
    ax6.scatter(fitted_vals, resid, alpha=0.35, s=8, color="#16a085")
    ax6.axhline(0, color="red", linestyle="--")
    try:
        from statsmodels.nonparametric.smoothers_lowess import lowess
        sm = lowess(resid, fitted_vals, frac=0.3)
        ax6.plot(sm[:, 0], sm[:, 1], color="#e67e22", linewidth=1.5)
    except Exception:
        pass
    ax6.set_title("Residuals vs Fitted")
    ax6.set_xlabel("Fitted"); ax6.set_ylabel("Residual")

    fig.suptitle(
        f"Residual Diagnostics — {label}\n"
        "Row 1: Time series & distribution  |  Row 2: ACF & PACF  |  Row 3: Q-Q & Residuals vs Fitted",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()


def print_residual_tests(residuals, label):
    from statsmodels.stats.diagnostic import acorr_ljungbox
    from scipy.stats import shapiro, anderson

    print(f"\n  {'─'*50}")
    print(f"  Residual diagnostics: {label}")
    print(f"  {'─'*50}")
    print(f"  Mean : {residuals.mean():.6f}")
    print(f"  SD   : {residuals.std():.4f}")

    try:
        lb = acorr_ljungbox(residuals, lags=[10, 20], return_df=True)
        for lag_val, row in lb.iterrows():
            flag = "✗ autocorrelated" if row["lb_pvalue"] < 0.05 else "✓ no autocorrelation"
            print(f"  Ljung-Box lag {int(lag_val):>2}: p={row['lb_pvalue']:.4f}  {flag}")
    except Exception as e:
        print(f"  Ljung-Box: {e}")

    if len(residuals) <= 5000:
        stat, p = shapiro(residuals)
        name    = "Shapiro-Wilk"
    else:
        res  = anderson(residuals, dist="norm")
        stat = res.statistic
        p    = 0.05 if res.statistic > res.critical_values[2] else 0.10
        name = "Anderson-Darling"
    flag = "✗ non-normal" if p < 0.05 else "✓ approximately normal"
    print(f"  {name}: stat={stat:.4f}, p≈{p:.4f}  {flag}")

In [ ]:
def run_commodity(comm: str, sub: pd.DataFrame):
    print(f"\n{'='*60}\n  Commodity: {comm}\n{'='*60}")

    sub = sub.sort_values("date").reset_index(drop=True)

    if "split" not in sub.columns:
        raise ValueError("'split' column not found. Expected values: 'train' / 'test'.")

    train_sub = sub[sub["split"] == "train"].reset_index(drop=True)
    test_sub  = sub[sub["split"] == "test"].reset_index(drop=True)
    n_train   = len(train_sub)
    n_test    = len(test_sub)

    if n_train < LOOKBACK + 30:
        print("  !! Not enough training rows — skipping."); return None
    if n_test == 0:
        print("  !! No test rows found — skipping."); return None

    avail    = [c for c in FEATURE_COLS
                if c in sub.columns and train_sub[c].notna().sum() > LOOKBACK]
    all_cols = [TARGET] + [f for f in avail if f != TARGET]
    n_vars   = len(all_cols)
    print(f"  Variates ({n_vars}): {all_cols}")

    train_data = train_sub[all_cols].copy().ffill().fillna(train_sub[all_cols].mean()).values
    test_data  = test_sub[all_cols].copy().ffill().fillna(train_sub[all_cols].mean()).values

    # Context window: last LOOKBACK rows of train + all test rows
    test_ctx = np.concatenate([train_data[-LOOKBACK:], test_data], axis=0)

    print(f"  Train: {train_sub['date'].iloc[0]} -> {train_sub['date'].iloc[-1]} ({n_train} weeks)")
    print(f"  Test:  {test_sub['date'].iloc[0]}  -> {test_sub['date'].iloc[-1]}  ({n_test} weeks)")

    feat_scaler = StandardScaler().fit(train_data)
    tgt_scaler  = StandardScaler().fit(train_data[:, [0]])

    train_sc = feat_scaler.transform(train_data)
    test_sc  = feat_scaler.transform(test_ctx)

    n_val = max(int(n_train * 0.15), LOOKBACK + 5)
    n_tr  = n_train - n_val

    train_ds = TSDataset(train_sc[:n_tr],              train_sc[:n_tr, 0],              LOOKBACK, HORIZON)
    val_ds   = TSDataset(train_sc[n_tr - LOOKBACK:],   train_sc[n_tr - LOOKBACK:, 0],  LOOKBACK, HORIZON)

    if len(train_ds) < 2 or len(val_ds) < 2:
        print("  !! Dataset windows too small — skipping."); return None

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    model = MambaForecaster(
        n_vars=n_vars, seq_len=LOOKBACK, horizon=HORIZON,
        d_model=D_MODEL, d_state=D_STATE, d_conv=D_CONV,
        expand=EXPAND,   n_layers=N_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {n_params:,}")

    opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    criterion = nn.MSELoss()

    best_val, best_state, patience_cnt = float("inf"), None, 0
    tr_losses, vl_losses               = [], []

    print(f"\n  Training up to {EPOCHS} epochs (patience={PATIENCE})...")
    for epoch in range(1, EPOCHS + 1):
        tl       = train_epoch(model, train_loader, opt, criterion, DEVICE)
        vl, _, _ = eval_epoch(model, val_loader,   criterion, DEVICE)
        scheduler.step()
        tr_losses.append(tl); vl_losses.append(vl)

        if vl < best_val:
            best_val, best_state, patience_cnt = vl, deepcopy(model.state_dict()), 0
        else:
            patience_cnt += 1

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:>3}  train={tl:.5f}  val={vl:.5f}  best={best_val:.5f}")
        if patience_cnt >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}"); break

    best_ep = int(np.argmin(vl_losses))
    model.load_state_dict(best_state)

    pred_sc   = walk_forward(model, test_sc, n_test, LOOKBACK, DEVICE)
    pred_orig = tgt_scaler.inverse_transform(pred_sc.reshape(-1, 1)).flatten()
    actual    = test_sub[TARGET].values
    train_orig= train_sub[TARGET].values

    min_len   = min(len(pred_orig), len(actual))
    pred_orig = pred_orig[:min_len]
    actual    = actual[:min_len]

    metrics = compute_metrics(actual, pred_orig, train_orig, f"{comm} — Mamba")
    print(f"\n  METRICS:")
    for k, v in metrics.items():
        if k != "model":
            print(f"    {k:<8}: {v}")

    residuals   = actual - pred_orig
    dates_train = pd.to_datetime(train_sub["date"].values)
    dates_test  = pd.to_datetime(test_sub["date"].values[:min_len])

    print_residual_tests(residuals, f"{comm} — Mamba")

    show_training_curve(tr_losses, vl_losses, best_ep, f"{comm} — Mamba")
    show_forecast_plot(dates_train, train_orig, dates_test, actual, pred_orig, f"{comm} — Mamba")
    show_residual_diagnostics(residuals, pred_orig, f"{comm} — Mamba")

    return metrics

In [ ]:
TARGET_COMMODITIES = [
    "Beetroot",
    "bulb onion yellow",
    "cabbage",
    "carrot",
    "cucumber",
    "curly kale",
    "lettuce",
    "tomato round",
]

df = (
    pd.read_csv(INPUT_FILE, parse_dates=["date"])
    .query("commodity in @TARGET_COMMODITIES")
    .sort_values(["commodity", "date"])
    .reset_index(drop=True)
)

commodities = df["commodity"].unique()
print(f"Filtered commodities : {list(commodities)}")
print(f"Device               : {DEVICE}")
print(f"Lookback             : {LOOKBACK} weeks")

all_metrics = []
for comm in commodities:
    result = run_commodity(comm, df[df["commodity"] == comm].copy())
    if result:
        all_metrics.append(result)

if all_metrics:
    summary = pd.DataFrame(all_metrics)
    print("  FINAL METRICS SUMMARY")
    print(summary.to_string(index=False))
    summary.to_csv(os.path.join("outputs", "mamba_metrics_filtered.csv"), index=False)
    print("\n✅ Done. mamba_metrics_filtered.csv saved.")